In [ ]:
!pip install bertopic sentence-transformers kiwipiepy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

!find /content/drive/MyDrive -name "hns_processed.csv" 2>/dev/null

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/hns_processed.csv")
print(f"Total: {len(df)}")
print(df.columns.tolist())

In [ ]:
import ast
import warnings
warnings.filterwarnings("ignore")
from kiwipiepy import Kiwi
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Filter to HNS-relevant documents
HNS_KEYWORDS = {
    "비듬", "두피", "각질", "가려움", "지루성",
    "설페이트", "클리니컬", "프로페셔널", "차콜",
    "약산성", "안티트로", "두피염", "정수리"
}
df_hns = df[df["raw_text"].apply(
    lambda x: any(kw in str(x) for kw in HNS_KEYWORDS)
)].copy().reset_index(drop=True)
print(f"Filtered: {len(df_hns)}")

docs = df_hns["raw_text"].fillna("").tolist()
docs = [d for d in docs if len(d) >= 10]
print(f"Docs: {len(docs)}")

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

kiwi = Kiwi()
kiwi.add_user_word("안티트로", "NNP", 0)

def korean_tokenizer(text):
    try:
        result = kiwi.analyze(text)
        tokens = [
            token.form for token in result[0][0]
            if token.tag in ("NNG", "NNP") and len(token.form) >= 2
        ]
        return tokens if tokens else text.split()
    except:
        return text.split()

vectorizer = CountVectorizer(
    tokenizer=korean_tokenizer,
    min_df=2,
    max_df=0.9,
    max_features=3000
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    representation_model=KeyBERTInspired(),
    nr_topics="auto",
    min_topic_size=8,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
print(f"\nTopics detected: {len(set(topics)) - 1}")

In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info[topic_info["Topic"] != -1][["Topic", "Count", "Name"]].to_string(index=False))

for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    keywords = topic_model.get_topic(topic_id)
    kw_str = ", ".join([kw for kw, _ in keywords[:8]])
    count = topics.count(topic_id)
    print(f"Topic {topic_id} ({count}): {kw_str}")

In [ ]:
import json

# LDA topics confirmed from hns_lda_results
LDA_TOPICS = {
    "클리니컬_스트렝스": ["클리니컬", "스트렝스", "비듬", "효과"],
    "정수리_냄새": ["정수리", "냄새", "차콜", "클린"],
    "지루성_두피염": ["지루성두피염", "지루", "각질", "증상"],
    "경쟁_이탈": ["안티트로", "니조랄", "대신", "갈아탔"],
}

consensus = []
for lda_name, lda_kws in LDA_TOPICS.items():
    print(f"[ LDA: {lda_name} ]")
    found = False
    for topic_id in sorted(set(topics)):
        if topic_id == -1:
            continue
        bert_kws = [kw for kw, _ in topic_model.get_topic(topic_id)[:10]]
        overlap = set(lda_kws) & set(bert_kws)
        if overlap:
            count = topics.count(topic_id)
            print(f"  BERTopic Topic {topic_id} ({count}) | {overlap}")
            consensus.append({
                "lda_topic": lda_name,
                "bertopic_id": topic_id,
                "overlap_keywords": list(overlap),
                "bertopic_count": count,
                "confidence": "high"
            })
            found = True
    if not found:
        print(f"  No BERTopic match (LDA-only signal)")
        consensus.append({
            "lda_topic": lda_name,
            "bertopic_id": None,
            "overlap_keywords": [],
            "bertopic_count": 0,
            "confidence": "low"
        })
    print()

In [ ]:
topic_info_filtered = topic_info[topic_info["Topic"] != -1]
topic_info_filtered.to_csv("hns_bertopic_results.csv",
                            index=False, encoding="utf-8-sig")

df_hns_result = df_hns.iloc[:len(topics)].copy()
df_hns_result["bertopic_id"] = topics
df_hns_result.to_csv("hns_bertopic_documents.csv",
                      index=False, encoding="utf-8-sig")

topic_keywords = {}
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    topic_keywords[str(tid)] = [kw for kw, _ in topic_model.get_topic(tid)[:10]]

with open("hns_bertopic_keywords.json", "w", encoding="utf-8") as f:
    json.dump(topic_keywords, f, ensure_ascii=False, indent=2)

consensus_df = pd.DataFrame(consensus)
consensus_df.to_csv("hns_lda_bertopic_consensus.csv",
                     index=False, encoding="utf-8-sig")

from google.colab import files
files.download("hns_bertopic_results.csv")
files.download("hns_bertopic_documents.csv")
files.download("hns_bertopic_keywords.json")
files.download("hns_lda_bertopic_consensus.csv")